# Step 01 — one tidy metadata table, and stage derived from SLEDAI

**Data type: RNA_array** (GSE65391). **Reads:** the series matrix. **Writes:** `step01_metadata.rds`.

GEO stores each sample characteristic as `key: value`. This step keeps the fields later steps use,
gives them plain names and proper types, and turns `"Not Applicable"`, `"Data Not Available"`,
`"N/A"` and `"Unk."` into missing values.

**Stage is not in the data; we derive it.** The SLEDAI (SLE Disease Activity Index) score of each
visit is grouped into five categories: none (0), mild (1–5), moderate (6–10), high (11–19) and very
high (20 or more).

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
suppressMessages({library(GEOquery); library(Biobase)})
es <- suppressMessages(getGEO(filename = raw("GSE65391", "GSE65391_series_matrix.txt.gz"), getGPL = FALSE))
pd <- pData(es)
c(samples = nrow(pd), characteristic_fields = length(grep(":ch1$", names(pd))))

samples characteristic_fields 
                  996                    87

In [2]:
ch <- function(key) {
  v <- as.character(pd[[paste0(key, ":ch1")]])
  v[v %in% c("Not Applicable", "Data Not Available", "NA", "N/A", "Unk.", "")] <- NA
  v
}
num <- function(key) suppressWarnings(as.numeric(ch(key)))

meta <- data.frame(
  sample   = rownames(pd),
  subject  = ch("subject"),
  disease  = factor(ch("disease state"), levels = c("Healthy", "SLE")),
  visit    = as.integer(ch("visit")),
  batch    = factor(ch("batch")),
  set      = ch("set"),
  age      = num("age"),
  sex      = ch("gender"),
  race     = ch("race"),
  sledai   = num("sledai"),
  # nephritis_class arrives as two fields joined by ";", for example
  # "Data Not Available;Prolif". The class is the second one.
  nephritis_class = sub(".*;", "", ch("nephritis_class")),
  # the authors' own molecular groups (Banchereau et al. 2016): a comparison
  # for our endotypes, never an input
  mdg      = ch("mdg"),
  cumulative_time = num("cumulative_time"),
  c3 = num("c3"), c4 = num("c4"), ds_dna = num("ds_dna"),
  wbc = num("wbc"), neutrophil_count = num("neutrophil_count"),
  lymphocyte_count = num("lymphocyte_count"),
  stringsAsFactors = FALSE, row.names = rownames(pd))
meta$nephritis_class[meta$nephritis_class %in% c("Not Applicable", "Data Not Available")] <- NA
meta$stage <- sledai_category(meta$sledai)
meta$stage[meta$disease == "Healthy"] <- NA
addmargins(table(disease = meta$disease, batch = meta$batch))

,1,2,Sum
Healthy,32,40,72
SLE,806,118,924
Sum,838,158,996


In [3]:
c(SLE_patients     = length(unique(meta$subject[meta$disease == "SLE"])),
  healthy_children = length(unique(meta$subject[meta$disease == "Healthy"])),
  SLE_samples      = sum(meta$disease == "SLE"))
table(stage = meta$stage, useNA = "ifany")
table(nephritis_class = meta$nephritis_class, useNA = "ifany")
table(mdg = meta$mdg, useNA = "ifany")

SLE_patients healthy_children      SLE_samples 
             158               46              924

stage
     none      mild  moderate      high very high      <NA> 
       93       390       277       127        37        72 

nephritis_class
      Membr       Mesan        NoLN Proli+Membr      Prolif        <NA> 
         42           8         498           3         238         207 

mdg
   0    1    2    3    4    5    6    7    8 <NA> 
  61  241   60   24    6    5    2    1    1  595 

## Visits are not time points

Each child was seen on their own schedule. `visit` counts visits within a child: visit 3 for one child
is not the same day, or the same stage of disease, as visit 3 for another. Stage is therefore taken
from each visit's SLEDAI, never from the visit number.

In [4]:
vp <- table(meta$subject[meta$disease == "SLE"])
rbind(visits_per_child = summary(as.integer(vp)),
      days_in_study    = summary(meta$cumulative_time[meta$disease == "SLE"]))

,Min.,1st Qu.,Median,Mean,3rd Qu.,Max.
visits_per_child,1,2.00,5,5.848101,8.0,22
days_in_study,0,77.75,258,357.493506,572.5,1412


In [5]:
saveRDS(meta, art("step01_metadata.rds"))
cat("wrote", art("step01_metadata.rds"), "\n")

wrote /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/data/run_artifacts/step01_metadata.rds 


## Findings

158 children with SLE contribute 924 samples; 46 healthy children contribute 72, of which 24 are
technical replicates (step 03). Children were seen a median of 5 times (up to 22) over up to 1,412
days. Every SLE visit has a SLEDAI category, most of them mild (390) or moderate (277). The nephritis
class is known for 789 of the 924 SLE samples: 498 with no lupus nephritis and 238 proliferative. The
authors' molecular group (`mdg`) is recorded for 401 samples, most of them in group 1.